# KAVACH: deep-learning router (Kaggle, GPU T4 x2, about 45 min)

**Before running**
1. Upload **`kavach_kaggle.zip`** as a new Kaggle **Dataset** (name it `kavach`). Kaggle unzips it automatically.
2. In this notebook: **Add Input → Your datasets → kavach**.
3. **Settings:** Accelerator **GPU T4 x2**, Internet **On** (needed to download the pretrained models from Hugging Face).
4. **Save Version → Save & Run All**, or run the cells one by one.
5. When it finishes, download **`kavach_dl_outputs.zip`** from the Output panel and unzip it into your `kavach` folder.

**What it does**
- Fine-tunes three multilingual transformers as KAVACH's *router*: XLM-R base, MuRIL base and DistilmBERT. XLM-R and MuRIL use 3 seeds each.
- Trains a TF-IDF + logistic-regression baseline.
- Tests generalisation: routers are trained *without* the India-safety data, then tested on it.
- Re-runs the end-to-end Gemini cascade with each learned router. It uses your cached Gemini answers, so it needs **0 API calls** and no key.


In [ ]:
# 1. SETUP -------------------------------------------------------------------
import glob, json, os, shutil, subprocess, sys, time
src = glob.glob('/kaggle/input/**/kavach/router.py', recursive=True)
assert src, 'Add the kavach dataset to this notebook (Add Input -> Your datasets)'
root = os.path.dirname(os.path.dirname(src[0]))
W = '/kaggle/working/kavach'
shutil.copytree(root, W, dirs_exist_ok=True)
os.chdir(W); sys.path.insert(0, W)
for d in ['dl/runs', 'dl/logs', 'dl/report']:
    os.makedirs(d, exist_ok=True)
print('project at', W)
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], capture_output=True, text=True).stdout)
import torch, transformers, sklearn
print('torch', torch.__version__, '| transformers', transformers.__version__, '| GPUs', torch.cuda.device_count())
print('cache present:', os.path.exists('bench/results/llm_cache.sqlite'), '| judge results present:', os.path.exists('bench/results/test_llm.json'))


## 2. Classical baseline: TF-IDF + logistic regression (CPU, about 1 min)

In [ ]:
!python dl/tfidf_baseline.py --out dl/runs/tfidf | tail -1
!python dl/tfidf_baseline.py --exclude_source india-safety --out dl/runs/tfidf_loso | tail -1


## 3. Fine-tune the transformers on both GPUs (about 30 min)

In [ ]:
MODELS = {'xlmr': 'xlm-roberta-base', 'muril': 'google/muril-base-cased', 'mbert': 'distilbert-base-multilingual-cased'}
jobs = []
for tag, seeds in [('xlmr', [13, 21, 42]), ('muril', [13, 21, 42]), ('mbert', [13])]:
    for s in seeds:
        jobs.append((f'{tag}_s{s}', MODELS[tag], s, None))
for tag in ['xlmr', 'muril']:                                    # generalisation to an unseen source
    jobs.append((f'{tag}_loso', MODELS[tag], 13, 'india-safety'))
jobs = [j for j in jobs if not os.path.exists(f'dl/runs/{j[0]}/router_config.json')]
ngpu = max(torch.cuda.device_count(), 1)
queues = [jobs[i::ngpu] for i in range(ngpu)]
procs = []
for g, q in enumerate(queues):
    if not q: continue
    cmd = ' ; '.join(
        f'python dl/train_router.py --model {m} --seed {s} --epochs 4 --bs 16 --lr 2e-5 --max_len 256 '
        + (f'--exclude_source {ex} ' if ex else '') + f'--out dl/runs/{name} > dl/logs/{name}.log 2>&1'
        for name, m, s, ex in q)
    procs.append(subprocess.Popen(cmd, shell=True, env={**os.environ, 'CUDA_VISIBLE_DEVICES': str(g), 'TOKENIZERS_PARALLELISM': 'false'}))
    print(f'GPU {g}:', [j[0] for j in q])
while any(p.poll() is None for p in procs):
    time.sleep(60)
    done = sorted(os.path.basename(os.path.dirname(p)) for p in glob.glob('dl/runs/*/router_config.json'))
    print(time.strftime('%H:%M'), 'finished:', done, flush=True)
for f in sorted(glob.glob('dl/logs/*.log')):
    lines = open(f).read().strip().splitlines()
    print('=====', f); print('\n'.join(l for l in lines if l.startswith('{') or l.startswith('FINAL') or 'Error' in l)[-1500:])


## 4. Analysis: routing recall and cost, ROC, languages, unseen source

In [ ]:
!python dl/analyze.py --runs dl/runs --out dl/report
from IPython.display import Image, display
for f in ['fig_router_roc.png', 'fig_router_lang.png', 'fig_training.png']:
    if os.path.exists(f'dl/report/{f}'):
        display(Image(f'dl/report/{f}', width=650))


## 5. End-to-end cascade with the learned routers (served from the Gemini cache: 0 API calls)

In [ ]:
# The cascade re-uses your cached Gemini answers, but a few multi-turn turns are reached ONLY in cascade mode
# (always-judge mode stopped earlier on those conversations), so they are not in the cache.
# Add your key once:  Add-ons -> Secrets -> name GEMINI_API_KEY  (about 10-30 real API calls per router)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['GEMINI_API_KEY'] = UserSecretsClient().get_secret('GEMINI_API_KEY')
    print('using GEMINI_API_KEY secret for cache misses')
except Exception:
    print('!! no GEMINI_API_KEY secret: cache misses will count as judge failures (results slightly biased)')

# pick the best transformer seed per model family BY VALIDATION AUROC (no test data used for selection)
best = {}
for d in glob.glob('dl/runs/*_s*'):
    cfg = json.load(open(f'{d}/router_config.json'))
    fam = os.path.basename(d).split('_')[0]
    if fam not in best or cfg['best_val_auroc'] > best[fam][1]:
        best[fam] = (d, cfg['best_val_auroc'])
print({k: (os.path.basename(v[0]), round(v[1], 4)) for k, v in best.items()})
!python dl/cascade_eval.py --router dl/runs/tfidf --label tfidf 2>&1 | grep -E "^(group|all|judge-call|new API)"
for fam, (d, _) in best.items():
    print('=' * 20, fam)
    !python dl/cascade_eval.py --router {d} --label {fam} 2>&1 | grep -E "^(group|all|judge-call|new API)"


## 6. Paper tables + package outputs

In [ ]:
!python bench/make_tables.py
overall = max(best.items(), key=lambda kv: kv[1][1])          # best family by validation AUROC -> demo router
print('demo router:', overall[0], os.path.basename(overall[1][0]))
B = '/kaggle/working/bundle'
shutil.rmtree(B, ignore_errors=True)
for sub in ['dl/report', 'paper/tables', 'dl/logs']:
    shutil.copytree(sub, f'{B}/{sub}')
for d in glob.glob('dl/runs/*'):                                # metrics only, no weights
    os.makedirs(f'{B}/{d}', exist_ok=True)
    for f in ['router_config.json', 'history.json', 'test_preds.jsonl', 'val_preds.jsonl', 'model.joblib']:
        if os.path.exists(f'{d}/{f}'):
            shutil.copy(f'{d}/{f}', f'{B}/{d}/')
os.makedirs(f'{B}/bench/results', exist_ok=True)
for f in glob.glob('bench/results/test_llm_router_*.json') + ['bench/results/llm_cache.sqlite']:
    shutil.copy(f, f'{B}/bench/results/')
os.chdir('/kaggle/working')                                     # download links are relative to /kaggle/working
for old in ['kavach_results.zip', 'kavach_best_router.zip', 'kavach_dl_outputs.zip']:
    if os.path.exists(old): os.remove(old)
z1 = shutil.make_archive('kavach_results', 'zip', B)            # small: metrics, figures, tables, cache
shutil.rmtree(B)
B2 = '/kaggle/working/bundle2'
shutil.rmtree(B2, ignore_errors=True)
shutil.copytree(f'kavach/{overall[1][0]}', f'{B2}/dl/best_router')  # large: the transformer used in the demo
z2 = shutil.make_archive('kavach_best_router', 'zip', B2)
shutil.rmtree(B2)
for z in (z1, z2):
    print(f'{z}  ({os.path.getsize(z)/1e6:.0f} MB)')
from IPython.display import FileLink, display
display(FileLink('kavach_results.zip')); display(FileLink('kavach_best_router.zip'))
print('If the links do not open: right-hand panel -> Output -> /kaggle/working -> click the file -> Download')
os.chdir('/kaggle/working/kavach')
